<!--
SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: Apache-2.0
-->

# SampleMetadata Walkthrough

This notebook provides a comprehensive tutorial on the `SampleMetadata` class, which is a core component of the ai-tune library for tracking and managing tensor metadata in PyTorch models.

## Table of Contents
1. [Setup](#setup)
2. [Introduction to SampleMetadata](#introduction)
3. [Creating Metadata from Inputs](#creating-metadata)
4. [Strict vs Non-Strict Mode](#strict-mode)
5. [Working with Nested Structures](#nested-structures)
6. [Describing Metadata - InfoLevel](#info-levels)
7. [Dynamic Shape Tracking](#dynamic-shapes)
8. [Batch Manipulation](#batch-manipulation)
9. [TensorSpec Deep Dive](#tensor-spec)
10. [Practical Example - Complete Workflow](#practical-example)
11. [Summary](#summary)


## 1. Setup <a id='setup'></a>

First, let's set up our environment and import the necessary modules.


In [1]:
%load_ext autoreload
%autoreload 2
%cd ..


/home/jkisel/workspace/ai-tune


In [2]:
from dataclasses import dataclass

import torch

from aitune.torch.module.sample_metadata import SampleMetadata, InfoLevel

/home/jkisel/workspace/ai-tune/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/jkisel/workspace/ai-tune/.venv/lib/python3.12/site-packages/modelopt/torch/utils/logging.py:115: UserWarning: Failed to import diffusers plugin due to: RuntimeError("Failed to import diffusers.models.modeling_utils because of the following error (look up to see its traceback):\nname 'logger' is not defined"). You may ignore this warning if you do not need this plugin.
  warnings.warn(message, *args, **kwargs)


Neither TRTLLM_PLUGIN_PATH is set nor is it directed to download the shared library. Please set either of the two to use TRT-LLM libraries in torchTRT


[07/15/2026-17:44:38] [TRT] [W] Functionality provided through tensorrt.plugin module is experimental.


/home/jkisel/workspace/ai-tune/.venv/lib/python3.12/site-packages/tensorrt_bindings/plugin/_lib.py:620: SyntaxWarning: invalid escape sequence '\s'
  """
/home/jkisel/workspace/ai-tune/.venv/lib/python3.12/site-packages/tensorrt_bindings/plugin/_tensor.py:133: SyntaxWarning: invalid escape sequence '\s'
  """
/home/jkisel/workspace/ai-tune/.venv/lib/python3.12/site-packages/tensorrt_bindings/plugin/_tensor.py:265: SyntaxWarning: invalid escape sequence '\s'
  """
/home/jkisel/workspace/ai-tune/.venv/lib/python3.12/site-packages/tensorrt_bindings/plugin/_tensor.py:392: SyntaxWarning: invalid escape sequence '\s'
  """
/home/jkisel/workspace/ai-tune/.venv/lib/python3.12/site-packages/tensorrt_bindings/plugin/_tensor.py:450: SyntaxWarning: invalid escape sequence '\s'
  """
/home/jkisel/workspace/ai-tune/.venv/lib/python3.12/site-packages/tensorrt_bindings/plugin/_tensor.py:897: SyntaxWarning: invalid escape sequence '\s'
  """
/home/jkisel/workspace/ai-tune/.venv/lib/python3.12/site-pack

## 2. Introduction to SampleMetadata <a id='introduction'></a>

### What is SampleMetadata?

`SampleMetadata` is a class designed to capture and track metadata about function inputs and outputs, particularly focusing on PyTorch tensors. It serves several important purposes:

- **Tensor Tracking**: Automatically discovers and tracks all tensors in complex nested data structures
- **Shape Inference**: Learns about dynamic dimensions and batch axes by observing multiple samples
- **Model Optimization**: Enables optimization backends to understand input/output characteristics
- **Dynamic Batching**: Supports scaling tensors to different batch sizes based on learned patterns

### Key Concepts

1. **Locators**: Navigate through nested structures (tuples, lists, dicts, dataclasses and registered user types) to find tensors
2. **TensorSpec**: Underlying representation that tracks shape, dtype, and batch axis information
3. **Dynamic Dimensions**: Dimensions that vary across samples (e.g., sequence length in NLP)
4. **Batch Axes**: Dimensions that scale proportionally with batch size

Let's start with a simple example:


In [3]:
# Create a simple tensor and capture its metadata
simple_tensor = torch.randn(2, 3, 4)
inputs = {"input": simple_tensor}

metadata = SampleMetadata.from_inputs(inputs)
print(repr(metadata))


Tensors:
╒═══════════════╤═════════════════╤═══════════╤═════════════╤═════════════╤═══════════════╕
│ Access Path   │ Semantic Path   │ Shape     │ Min Shape   │ Max Shape   │ Dtype         │
╞═══════════════╪═════════════════╪═══════════╪═════════════╪═════════════╪═══════════════╡
│ input         │ input           │ [2, 3, 4] │ [2, 3, 4]   │ [2, 3, 4]   │ torch.float32 │
╘═══════════════╧═════════════════╧═══════════╧═════════════╧═════════════╧═══════════════╛



The output shows that `SampleMetadata` automatically detected the tensor under the forward parameter path `input` and captured its shape `[2, 3, 4]` along with data type information.


## 3. Creating Metadata from Inputs <a id='creating-metadata'></a>

The primary way to create `SampleMetadata` is through the `from_inputs()` static method. It accepts:

- `inputs`: Values keyed by their forward parameter names
- `strict`: Boolean flag controlling whether to track non-tensor data (default: `False`)
- `batch_size`: Optional batch size used to detect batch dimensions

AITune obtains this dictionary by normalizing a call with its forward signature. These examples provide the dictionary directly so they can focus on metadata behavior.


In [4]:
# Example 1: Multiple tensor parameters
inputs = {
    "x": torch.randn(2, 3),
    "y": torch.randn(4, 5),
}

meta1 = SampleMetadata.from_inputs(inputs)
print("Example 1 - Multiple tensor parameters:")
print(repr(meta1))


Example 1 - Multiple tensor parameters:
Tensors:
╒═══════════════╤═════════════════╤═════════╤═════════════╤═════════════╤═══════════════╕
│ Access Path   │ Semantic Path   │ Shape   │ Min Shape   │ Max Shape   │ Dtype         │
╞═══════════════╪═════════════════╪═════════╪═════════════╪═════════════╪═══════════════╡
│ x             │ x               │ [2, 3]  │ [2, 3]      │ [2, 3]      │ torch.float32 │
├───────────────┼─────────────────┼─────────┼─────────────┼─────────────┼───────────────┤
│ y             │ y               │ [4, 5]  │ [4, 5]      │ [4, 5]      │ torch.float32 │
╘═══════════════╧═════════════════╧═════════╧═════════════╧═════════════╧═══════════════╛



In [5]:
# Example 2: Named tensor parameters
inputs = {
    "input_tensor": torch.randn(3, 4),
    "mask": torch.randn(3, 1),
}

meta2 = SampleMetadata.from_inputs(inputs)
print("Example 2 - Named tensors:")
print(repr(meta2))


Example 2 - Named tensors:
Tensors:
╒═══════════════╤═════════════════╤═════════╤═════════════╤═════════════╤═══════════════╕
│ Access Path   │ Semantic Path   │ Shape   │ Min Shape   │ Max Shape   │ Dtype         │
╞═══════════════╪═════════════════╪═════════╪═════════════╪═════════════╪═══════════════╡
│ input_tensor  │ input_tensor    │ [3, 4]  │ [3, 4]      │ [3, 4]      │ torch.float32 │
├───────────────┼─────────────────┼─────────┼─────────────┼─────────────┼───────────────┤
│ mask          │ mask            │ [3, 1]  │ [3, 1]      │ [3, 1]      │ torch.float32 │
╘═══════════════╧═════════════════╧═════════╧═════════════╧═════════════╧═══════════════╛



In [6]:
# Example 3: Mixed primitives and tensors
inputs = {
    "label": "some_string",           # Primitive - ignored by default
    "weights": torch.randn(2, 2),     # Tensor - tracked
    "steps": 42,                      # Primitive - ignored by default
    "data": torch.randn(3, 3),
    "learning_rate": 0.001,           # Primitive - ignored by default
}

meta3 = SampleMetadata.from_inputs(inputs)
print("Example 3 - Mixed types (strict=False):")
print(repr(meta3))
print("\nNotice that only tensors are tracked!")


Example 3 - Mixed types (strict=False):
Tensors:
╒═══════════════╤═════════════════╤═════════╤═════════════╤═════════════╤═══════════════╕
│ Access Path   │ Semantic Path   │ Shape   │ Min Shape   │ Max Shape   │ Dtype         │
╞═══════════════╪═════════════════╪═════════╪═════════════╪═════════════╪═══════════════╡
│ weights       │ weights         │ [2, 2]  │ [2, 2]      │ [2, 2]      │ torch.float32 │
├───────────────┼─────────────────┼─────────┼─────────────┼─────────────┼───────────────┤
│ data          │ data            │ [3, 3]  │ [3, 3]      │ [3, 3]      │ torch.float32 │
╘═══════════════╧═════════════════╧═════════╧═════════════╧═════════════╧═══════════════╛


Notice that only tensors are tracked!


## 4. Strict vs Non-Strict Mode <a id='strict-mode'></a>

By default, `SampleMetadata` operates in **non-strict mode** (`strict=False`), which means it only tracks tensors and ignores all other data types. This is useful when you only care about tensor shapes for optimization purposes.

However, when `strict=True`, `SampleMetadata` also captures primitive values and other non-tensor data. This can be useful for:

- Validating that function signatures match expected patterns
- Debugging data flow through complex pipelines
- Ensuring reproducibility of function calls

Let's compare the two modes:


In [7]:
# Same inputs, different modes
inputs = {
    "values": (1, 2, 3, torch.randn(2, 2)),
    "t": torch.randn(2, 3),
    "other": "abc",
}

# Non-strict mode (default)
meta_non_strict = SampleMetadata.from_inputs(inputs, strict=False)
print("Non-Strict Mode (strict=False):")
print(repr(meta_non_strict))
print("\n" + "="*80 + "\n")

# Strict mode
meta_strict = SampleMetadata.from_inputs(inputs, strict=True)
print("Strict Mode (strict=True):")
print(repr(meta_strict))


Non-Strict Mode (strict=False):
Tensors:
╒═══════════════╤═════════════════╤═════════╤═════════════╤═════════════╤═══════════════╕
│ Access Path   │ Semantic Path   │ Shape   │ Min Shape   │ Max Shape   │ Dtype         │
╞═══════════════╪═════════════════╪═════════╪═════════════╪═════════════╪═══════════════╡
│ values[3]     │ ('values', 3)   │ [2, 2]  │ [2, 2]      │ [2, 2]      │ torch.float32 │
├───────────────┼─────────────────┼─────────┼─────────────┼─────────────┼───────────────┤
│ t             │ t               │ [2, 3]  │ [2, 3]      │ [2, 3]      │ torch.float32 │
╘═══════════════╧═════════════════╧═════════╧═════════════╧═════════════╧═══════════════╛



Strict Mode (strict=True):
Tensors:
╒═══════════════╤═════════════════╤═════════╤═════════════╤═════════════╤═══════════════╕
│ Access Path   │ Semantic Path   │ Shape   │ Min Shape   │ Max Shape   │ Dtype         │
╞═══════════════╪═════════════════╪═════════╪═════════════╪═════════════╪═══════════════╡
│ values[3]     │ ('

Notice that in strict mode, we see an additional "Other" section that includes the primitive values (1, 2, 3, and "abc").


## 5. Working with Nested Structures <a id='nested-structures'></a>

One of the most powerful features of `SampleMetadata` is its ability to handle deeply nested data structures. Real-world model inputs often involve complex combinations of:

- **Tuples and Lists**: For variable-length sequences
- **Dictionaries**: For named parameters
- **Dataclasses**: For structured configuration objects

`SampleMetadata` uses **Locators** to navigate these structures and find all tensors, no matter how deeply nested they are.

Let's create a complex nested example:


In [8]:
# Define a custom dataclass
@dataclass
class ModelInput:
    data: torch.Tensor
    metadata: str

# Create complex nested structure
inputs = {
    "values": [
        "first_arg",
        torch.randn(1),                                      # Simple tensor
        (torch.randn(2), torch.randn(3)),                    # Tuple of tensors
        {"t": torch.randn(4)},                              # Dict with tensor
        ModelInput(data=torch.randn(5), metadata="info"),   # Dataclass with tensor
    ],
    "t1": torch.randn(1, 1),
    "t2": [torch.randn(2, 2), torch.randn(3, 3)],          # List of tensors
    "t3": ModelInput(data=torch.randn(4, 4), metadata="xyz"),
    "last": "other",
}

nested_meta = SampleMetadata.from_inputs(inputs, strict=True)
print(repr(nested_meta))


Tensors:
╒════════════════╤═══════════════════════╤═════════╤═════════════╤═════════════╤═══════════════╕
│ Access Path    │ Semantic Path         │ Shape   │ Min Shape   │ Max Shape   │ Dtype         │
╞════════════════╪═══════════════════════╪═════════╪═════════════╪═════════════╪═══════════════╡
│ values[1]      │ ('values', 1)         │ [1]     │ [1]         │ [1]         │ torch.float32 │
├────────────────┼───────────────────────┼─────────┼─────────────┼─────────────┼───────────────┤
│ values[2][0]   │ ('values', 2, 0)      │ [2]     │ [2]         │ [2]         │ torch.float32 │
├────────────────┼───────────────────────┼─────────┼─────────────┼─────────────┼───────────────┤
│ values[2][1]   │ ('values', 2, 1)      │ [3]     │ [3]         │ [3]         │ torch.float32 │
├────────────────┼───────────────────────┼─────────┼─────────────┼─────────────┼───────────────┤
│ values[3]["t"] │ ('values', 3, 't')    │ [4]     │ [4]         │ [4]         │ torch.float32 │
├────────────────┼───

### Understanding Paths

The **Access Path** column uses Python-like access syntax rooted at the real forward parameter name:

- `values[1]`: Second element of `values` (0-indexed)
- `values[2][0]`: First element of the tuple at `values[2]`
- `values[3]["t"]`: Value at key `t` in the dictionary at `values[3]`
- `values[4].data`: The `data` attribute of the dataclass at `values[4]`
- `t2[0]`: First element of the `t2` list
- `t3.data`: The `data` attribute of the `t3` dataclass

The **Semantic Path** column shows the stable identity used to match and configure inputs, for example `("values", 3, "t")`. The paired `Locator` also retains the container access information needed to retrieve and manipulate each value. Access paths are used in reports and as backend tensor IDs.


## 6. Describing Metadata - InfoLevel <a id='info-levels'></a>

`SampleMetadata` provides three levels of detail when displaying information, controlled by the `InfoLevel` enum:

1. **`InfoLevel.SHORT`**: Compact representation with tensor access paths
2. **`InfoLevel.MEDIUM`**: Includes access paths, semantic paths, and current shapes (simple table format)
3. **`InfoLevel.FULL`**: Complete details including both paths, min/max shapes, and dtypes (fancy table format)

Let's see the same metadata displayed at all three levels:


In [9]:
# Create sample metadata
inputs = {
    "x": torch.randn(2, 3),
    "hidden_states": torch.randn(4, 5, 6),
    "mask": torch.randn(2, 1),
}
meta = SampleMetadata.from_inputs(inputs)

print("InfoLevel.SHORT:")
print(meta.describe(InfoLevel.SHORT))
print("\n" + "="*80 + "\n")

print("InfoLevel.MEDIUM:")
print(meta.describe(InfoLevel.MEDIUM))
print("\n" + "="*80 + "\n")

print("InfoLevel.FULL:")
print(meta.describe(InfoLevel.FULL))


InfoLevel.SHORT:
Tensors: x, hidden_states, mask


InfoLevel.MEDIUM:
Tensors:
Path           Shape
-------------  ---------
x              [2, 3]
hidden_states  [4, 5, 6]
mask           [2, 1]



InfoLevel.FULL:
Tensors:
╒═══════════════╤═════════════════╤═══════════╤═════════════╤═════════════╤═══════════════╕
│ Access Path   │ Semantic Path   │ Shape     │ Min Shape   │ Max Shape   │ Dtype         │
╞═══════════════╪═════════════════╪═══════════╪═════════════╪═════════════╪═══════════════╡
│ x             │ x               │ [2, 3]    │ [2, 3]      │ [2, 3]      │ torch.float32 │
├───────────────┼─────────────────┼───────────┼─────────────┼─────────────┼───────────────┤
│ hidden_states │ hidden_states   │ [4, 5, 6] │ [4, 5, 6]   │ [4, 5, 6]   │ torch.float32 │
├───────────────┼─────────────────┼───────────┼─────────────┼─────────────┼───────────────┤
│ mask          │ mask            │ [2, 1]    │ [2, 1]      │ [2, 1]      │ torch.float32 │
╘═══════════════╧═════════════════╧════════

The `FULL` level is particularly useful because it shows:
- **Min Shape**: The smallest dimensions seen for each axis
- **Max Shape**: The largest dimensions seen for each axis
- **Dtype**: The PyTorch data type of the tensor

These become interesting when we start tracking multiple samples with different shapes.


## 7. Dynamic Shape Tracking <a id='dynamic-shapes'></a>

One of the most sophisticated features of `SampleMetadata` is its ability to learn about **dynamic dimensions** and **batch axes** by observing multiple samples with different shapes.

### How It Works

When you call `update_shapes_seen()` with metadata from a different sample:

1. **Batch Axis Detection**: If a dimension scales proportionally with batch size and the multiplier is an integer, it's marked as a batch axis (e.g., `batch0`, `batch1`)
2. **Dynamic Dimension Detection**: If a dimension changes but not proportionally to batch size, it's marked as a dynamic dimension (e.g., `dim0`, `dim1`)
3. **Min/Max Tracking**: The minimum and maximum values seen for each dimension are tracked

Let's see this in action:


In [10]:
# Create initial metadata with batch size 1
inputs_initial = {
    "features": [
        torch.randn(1),
        torch.randn(2),
        torch.randn(5),
    ],
    "data": torch.randn(1, 10),  # First dim batch, second dynamic
}

meta_initial = SampleMetadata.from_inputs(inputs_initial, strict=False, batch_size=1)
print("Initial Metadata (batch_size=1):")
print(meta_initial.describe(InfoLevel.FULL))


Initial Metadata (batch_size=1):
Tensors:
╒═══════════════╤═════════════════╤═════════╤═════════════╤═════════════╤═══════════════╕
│ Access Path   │ Semantic Path   │ Shape   │ Min Shape   │ Max Shape   │ Dtype         │
╞═══════════════╪═════════════════╪═════════╪═════════════╪═════════════╪═══════════════╡
│ features[0]   │ ('features', 0) │ [1]     │ [1]         │ [1]         │ torch.float32 │
├───────────────┼─────────────────┼─────────┼─────────────┼─────────────┼───────────────┤
│ features[1]   │ ('features', 1) │ [2]     │ [2]         │ [2]         │ torch.float32 │
├───────────────┼─────────────────┼─────────┼─────────────┼─────────────┼───────────────┤
│ features[2]   │ ('features', 2) │ [5]     │ [5]         │ [5]         │ torch.float32 │
├───────────────┼─────────────────┼─────────┼─────────────┼─────────────┼───────────────┤
│ data          │ data            │ [1, 10] │ [1, 10]     │ [1, 10]     │ torch.float32 │
╘═══════════════╧═════════════════╧═════════╧═════════════

In [11]:
# Create second metadata with different shapes and batch size 2
inputs_second = {
    "features": [
        torch.randn(2),      # Doubled (batch axis)
        torch.randn(5),      # Changed but not proportionally (dynamic)
        torch.randn(15),     # Changed but not proportionally (dynamic)
    ],
    "data": torch.randn(2, 25),  # First dim doubled, second changed
}

meta_second = SampleMetadata.from_inputs(inputs_second, strict=False, batch_size=2)
print("Second Metadata (batch_size=2):")
print(meta_second.describe(InfoLevel.FULL))


Second Metadata (batch_size=2):
Tensors:
╒═══════════════╤═════════════════╤═════════╤═════════════╤═════════════╤═══════════════╕
│ Access Path   │ Semantic Path   │ Shape   │ Min Shape   │ Max Shape   │ Dtype         │
╞═══════════════╪═════════════════╪═════════╪═════════════╪═════════════╪═══════════════╡
│ features[0]   │ ('features', 0) │ [2]     │ [2]         │ [2]         │ torch.float32 │
├───────────────┼─────────────────┼─────────┼─────────────┼─────────────┼───────────────┤
│ features[1]   │ ('features', 1) │ [5]     │ [5]         │ [5]         │ torch.float32 │
├───────────────┼─────────────────┼─────────┼─────────────┼─────────────┼───────────────┤
│ features[2]   │ ('features', 2) │ [15]    │ [15]        │ [15]        │ torch.float32 │
├───────────────┼─────────────────┼─────────┼─────────────┼─────────────┼───────────────┤
│ data          │ data            │ [2, 25] │ [2, 25]     │ [2, 25]     │ torch.float32 │
╘═══════════════╧═════════════════╧═════════╧═════════════╧

In [12]:
# Update the initial metadata with information from the second sample
meta_initial.update_shapes_seen(meta_second)
print("Updated Metadata (after seeing both samples):")
print(meta_initial.describe(InfoLevel.FULL))


Updated Metadata (after seeing both samples):
Tensors:
╒═══════════════╤═════════════════╤════════════════════╤═════════════╤═════════════╤═══════════════╕
│ Access Path   │ Semantic Path   │ Shape              │ Min Shape   │ Max Shape   │ Dtype         │
╞═══════════════╪═════════════════╪════════════════════╪═════════════╪═════════════╪═══════════════╡
│ features[0]   │ ('features', 0) │ ['batch0']         │ [1]         │ [2]         │ torch.float32 │
├───────────────┼─────────────────┼────────────────────┼─────────────┼─────────────┼───────────────┤
│ features[1]   │ ('features', 1) │ ['dim0']           │ [2]         │ [5]         │ torch.float32 │
├───────────────┼─────────────────┼────────────────────┼─────────────┼─────────────┼───────────────┤
│ features[2]   │ ('features', 2) │ ['dim0']           │ [5]         │ [15]        │ torch.float32 │
├───────────────┼─────────────────┼────────────────────┼─────────────┼─────────────┼───────────────┤
│ data          │ data            │ 

### Understanding the Results

After updating, notice how the shapes have been transformed:

- **`batch0`**: Dimensions that doubled when batch size doubled (1→2)
- **`dim0`, `dim1`**: Dimensions that changed but not proportionally to batch size
- **Min/Max Shape**: Now show the range of values observed

This information is crucial for:
- **Model compilation**: Backends can create optimized graphs for dynamic shapes
- **Memory planning**: Knowing the range helps allocate appropriate buffers
- **Validation**: Ensuring new inputs fall within expected ranges


## 8. Batch Manipulation <a id='batch-manipulation'></a>

Once `SampleMetadata` has learned about batch axes through `update_shapes_seen()`, it can use the `make_batch()` method to scale tensors to a target batch size.

### How `make_batch()` Works

The method uses **batch axis multipliers** to determine how to scale each dimension:

1. **Multiplier = 1**: Standard batch axis, scales linearly with batch size
2. **Multiplier > 1**: Stacked batch axis (e.g., when inputs are vertically stacked)
3. **Slicing**: If current size > target, slice the tensor
4. **Repeating**: If current size < target, repeat the tensor

Let's see this in action:


In [13]:
# First, create metadata and teach it about batch axes
inputs1 = {
    "features": [torch.randn(1, 5), torch.randn(2, 3)],
    "mask": torch.randn(1, 10),
}
meta = SampleMetadata.from_inputs(inputs1, batch_size=1)

# Second sample with batch size 2
inputs2 = {
    "features": [torch.randn(2, 5), torch.randn(4, 3)],
    "mask": torch.randn(2, 10),
}
meta2 = SampleMetadata.from_inputs(inputs2, batch_size=2)
meta.update_shapes_seen(meta2)

print("Learned Metadata:")
print(meta.describe(InfoLevel.FULL))


Learned Metadata:
Tensors:
╒═══════════════╤═════════════════╤════════════════╤═════════════╤═════════════╤═══════════════╕
│ Access Path   │ Semantic Path   │ Shape          │ Min Shape   │ Max Shape   │ Dtype         │
╞═══════════════╪═════════════════╪════════════════╪═════════════╪═════════════╪═══════════════╡
│ features[0]   │ ('features', 0) │ ['batch0', 5]  │ [1, 5]      │ [2, 5]      │ torch.float32 │
├───────────────┼─────────────────┼────────────────┼─────────────┼─────────────┼───────────────┤
│ features[1]   │ ('features', 1) │ ['batch0', 3]  │ [2, 3]      │ [4, 3]      │ torch.float32 │
├───────────────┼─────────────────┼────────────────┼─────────────┼─────────────┼───────────────┤
│ mask          │ mask            │ ['batch0', 10] │ [1, 10]     │ [2, 10]     │ torch.float32 │
╘═══════════════╧═════════════════╧════════════════╧═════════════╧═════════════╧═══════════════╛



In [14]:
# Now use make_batch to scale to a larger batch size
original_inputs = {
    "features": [torch.randn(1, 5), torch.randn(2, 3)],
    "mask": torch.randn(1, 10),
}

print("Original shapes:")
print(f"  features[0]: {original_inputs['features'][0].shape}")
print(f"  features[1]: {original_inputs['features'][1].shape}")
print(f"  mask: {original_inputs['mask'].shape}")
print()

# Scale to batch size 10
batched_inputs = meta.make_batch(original_inputs, batch_size=10)

print("After make_batch(batch_size=10):")
print(f"  features[0]: {batched_inputs['features'][0].shape}")
print(f"  features[1]: {batched_inputs['features'][1].shape}")
print(f"  mask: {batched_inputs['mask'].shape}")


Original shapes:
  features[0]: torch.Size([1, 5])
  features[1]: torch.Size([2, 3])
  mask: torch.Size([1, 10])

After make_batch(batch_size=10):
  features[0]: torch.Size([10, 5])
  features[1]: torch.Size([20, 3])
  mask: torch.Size([10, 10])


Notice how:
- The first dimensions (batch axes) scaled to match the target batch size of 10
- `features[1]` has a multiplier of 2 (it is a stacked batch), so it scaled to 20 (10 × 2)
- Non-batch dimensions (like the 5, 3, 10) remained unchanged


## 9. TensorSpec Deep Dive <a id='tensor-spec'></a>

`SampleMetadata` is a container for `(Locator, TensorSpec)` pairs. The `Locator` identifies the tensor by its forward parameter path, while `TensorSpec` stores tensor properties.

### TensorSpec Attributes

- **`shape`**: Current shape representation (may include symbolic dimensions)
- **`min_shape`**: Minimum dimensions observed
- **`max_shape`**: Maximum dimensions observed
- **`dtype`**: PyTorch data type
- **`_bs_multipliers`**: Internal batch size multipliers for each axis

Let's inspect tensor paths and their `TensorSpec` objects directly:


In [15]:
# Create metadata with dynamic shapes
inputs1 = {
    "features": [torch.randn(1, 5)],
    "data": torch.randn(1, 10),
}
meta = SampleMetadata.from_inputs(inputs1, batch_size=1)

inputs2 = {
    "features": [torch.randn(2, 5)],
    "data": torch.randn(2, 20),
}
meta2 = SampleMetadata.from_inputs(inputs2, batch_size=2)
meta.update_shapes_seen(meta2)

# Access individual TensorSpec objects
print("Individual TensorSpec objects:\n")
for locator, tensor_spec in meta.tensor_data:
    print(f"Access Path: {locator.display_path}")
    print(f"Semantic Path: {locator.path}")
    print(f"  Shape: {tensor_spec.shape}")
    print(f"  Min Shape: {tensor_spec.min_shape}")
    print(f"  Max Shape: {tensor_spec.max_shape}")
    print(f"  Dtype: {tensor_spec.dtype}")
    print(f"  Has batch axis: {tensor_spec.has_batch_axis()}")
    print(f"  Has dynamic axis: {tensor_spec.has_dynamic_axis()}")
    print(f"  Batch multipliers: {tensor_spec.get_batch_axis_multipliers()}")


Individual TensorSpec objects:

Access Path: features[0]
Semantic Path: ('features', 0)
  Shape: ['batch0', 5]
  Min Shape: [1, 5]
  Max Shape: [2, 5]
  Dtype: torch.float32
  Has batch axis: True
  Has dynamic axis: False
  Batch multipliers: {0: 1}
Access Path: data
Semantic Path: data
  Shape: ['batch0', 'batch1']
  Min Shape: [1, 10]
  Max Shape: [2, 20]
  Dtype: torch.float32
  Has batch axis: True
  Has dynamic axis: False
  Batch multipliers: {0: 1, 1: 10}


### Useful TensorSpec Methods

- **`has_batch_axis()`**: Returns True if the tensor has at least one batch dimension
- **`has_dynamic_axis()`**: Returns True if the tensor has at least one dynamic dimension
- **`get_batch_axis_multipliers()`**: Returns a dict mapping axis index to its batch multiplier
- **`matches(other)`**: Checks if two TensorSpecs are compatible

These methods are used internally by `SampleMetadata` to perform operations like `make_batch()`.


## 10. Practical Example - Complete Workflow <a id='practical-example'></a>

Let's put everything together with a realistic scenario: profiling a model with variable-length sequences (like in NLP tasks).

### Scenario

We have a language model that takes:
- Input IDs with shape `(batch_size, sequence_length)`
- Attention mask with shape `(batch_size, sequence_length)`
- Position IDs with shape `(batch_size, sequence_length)`

We'll profile it with different batch sizes and sequence lengths to learn the dynamic shapes.


In [16]:
# Simulate model profiling
@dataclass
class ModelInputs:
    input_ids: torch.Tensor
    attention_mask: torch.Tensor
    position_ids: torch.Tensor

# Sample 1: batch_size=1, seq_len=10
sample1_inputs = {
    "inputs": ModelInputs(
        input_ids=torch.randint(0, 1000, (1, 10)),
        attention_mask=torch.ones(1, 10),
        position_ids=torch.arange(10).unsqueeze(0),
    )
}

metadata = SampleMetadata.from_inputs(sample1_inputs, batch_size=1, strict=False)
print("After Sample 1 (batch=1, seq_len=10):")
print(metadata.describe(InfoLevel.FULL))


After Sample 1 (batch=1, seq_len=10):
Tensors:
╒═══════════════════════╤══════════════════════════════╤═════════╤═════════════╤═════════════╤═══════════════╕
│ Access Path           │ Semantic Path                │ Shape   │ Min Shape   │ Max Shape   │ Dtype         │
╞═══════════════════════╪══════════════════════════════╪═════════╪═════════════╪═════════════╪═══════════════╡
│ inputs.input_ids      │ ('inputs', 'input_ids')      │ [1, 10] │ [1, 10]     │ [1, 10]     │ torch.int64   │
├───────────────────────┼──────────────────────────────┼─────────┼─────────────┼─────────────┼───────────────┤
│ inputs.attention_mask │ ('inputs', 'attention_mask') │ [1, 10] │ [1, 10]     │ [1, 10]     │ torch.float32 │
├───────────────────────┼──────────────────────────────┼─────────┼─────────────┼─────────────┼───────────────┤
│ inputs.position_ids   │ ('inputs', 'position_ids')   │ [1, 10] │ [1, 10]     │ [1, 10]     │ torch.int64   │
╘═══════════════════════╧══════════════════════════════╧═════════

In [17]:
# Sample 2: batch_size=2, seq_len=15
sample2_inputs = {
    "inputs": ModelInputs(
        input_ids=torch.randint(0, 1000, (2, 15)),
        attention_mask=torch.ones(2, 15),
        position_ids=torch.arange(15).unsqueeze(0).repeat(2, 1),
    )
}

metadata2 = SampleMetadata.from_inputs(sample2_inputs, batch_size=2, strict=False)
metadata.update_shapes_seen(metadata2)

print("After Sample 2 (batch=2, seq_len=15):")
print(metadata.describe(InfoLevel.FULL))


After Sample 2 (batch=2, seq_len=15):
Tensors:
╒═══════════════════════╤══════════════════════════════╤════════════════════╤═════════════╤═════════════╤═══════════════╕
│ Access Path           │ Semantic Path                │ Shape              │ Min Shape   │ Max Shape   │ Dtype         │
╞═══════════════════════╪══════════════════════════════╪════════════════════╪═════════════╪═════════════╪═══════════════╡
│ inputs.input_ids      │ ('inputs', 'input_ids')      │ ['batch0', 'dim1'] │ [1, 10]     │ [2, 15]     │ torch.int64   │
├───────────────────────┼──────────────────────────────┼────────────────────┼─────────────┼─────────────┼───────────────┤
│ inputs.attention_mask │ ('inputs', 'attention_mask') │ ['batch0', 'dim1'] │ [1, 10]     │ [2, 15]     │ torch.float32 │
├───────────────────────┼──────────────────────────────┼────────────────────┼─────────────┼─────────────┼───────────────┤
│ inputs.position_ids   │ ('inputs', 'position_ids')   │ ['batch0', 'dim1'] │ [1, 10]     │ [2, 15]

In [18]:
# Sample 3: batch_size=4, seq_len=20
sample3_inputs = {
    "inputs": ModelInputs(
        input_ids=torch.randint(0, 1000, (4, 20)),
        attention_mask=torch.ones(4, 20),
        position_ids=torch.arange(20).unsqueeze(0).repeat(4, 1),
    )
}

metadata3 = SampleMetadata.from_inputs(sample3_inputs, batch_size=4, strict=False)
metadata.update_shapes_seen(metadata3)

print("After Sample 3 (batch=4, seq_len=20):")
print(metadata.describe(InfoLevel.FULL))


After Sample 3 (batch=4, seq_len=20):
Tensors:
╒═══════════════════════╤══════════════════════════════╤════════════════════╤═════════════╤═════════════╤═══════════════╕
│ Access Path           │ Semantic Path                │ Shape              │ Min Shape   │ Max Shape   │ Dtype         │
╞═══════════════════════╪══════════════════════════════╪════════════════════╪═════════════╪═════════════╪═══════════════╡
│ inputs.input_ids      │ ('inputs', 'input_ids')      │ ['batch0', 'dim1'] │ [1, 10]     │ [4, 20]     │ torch.int64   │
├───────────────────────┼──────────────────────────────┼────────────────────┼─────────────┼─────────────┼───────────────┤
│ inputs.attention_mask │ ('inputs', 'attention_mask') │ ['batch0', 'dim1'] │ [1, 10]     │ [4, 20]     │ torch.float32 │
├───────────────────────┼──────────────────────────────┼────────────────────┼─────────────┼─────────────┼───────────────┤
│ inputs.position_ids   │ ('inputs', 'position_ids')   │ ['batch0', 'dim1'] │ [1, 10]     │ [4, 20]

### Analysis

After observing three samples with different batch sizes and sequence lengths:

- **First dimension**: Identified as `batch0` because it scaled proportionally (1→2→4)
- **Second dimension**: Identified as `dim1` because it varied dynamically (10→15→20)
- **Min/Max ranges**: Captured the observed ranges for both dimensions

This information can now be used by optimization backends to compile efficient code for these dynamic shapes.


In [19]:
# Now we can create inputs for any batch size!
test_inputs = {
    "inputs": ModelInputs(
        input_ids=torch.randint(0, 1000, (2, 12)),
        attention_mask=torch.ones(2, 12),
        position_ids=torch.arange(12).unsqueeze(0).repeat(2, 1),
    )
}

print("Original test input shapes:")
print(f"  input_ids: {test_inputs['inputs'].input_ids.shape}")
print(f"  attention_mask: {test_inputs['inputs'].attention_mask.shape}")
print(f"  position_ids: {test_inputs['inputs'].position_ids.shape}")
print()

# Scale to batch size 8
scaled_inputs = metadata.make_batch(test_inputs, batch_size=8)

print("After scaling to batch_size=8:")
print(f"  input_ids: {scaled_inputs['inputs'].input_ids.shape}")
print(f"  attention_mask: {scaled_inputs['inputs'].attention_mask.shape}")
print(f"  position_ids: {scaled_inputs['inputs'].position_ids.shape}")
print("\nNote: Batch dimension scaled to 8, but sequence length (dim1) remained at 12")


Original test input shapes:
  input_ids: torch.Size([2, 12])
  attention_mask: torch.Size([2, 12])
  position_ids: torch.Size([2, 12])

After scaling to batch_size=8:
  input_ids: torch.Size([8, 12])
  attention_mask: torch.Size([8, 12])
  position_ids: torch.Size([8, 12])

Note: Batch dimension scaled to 8, but sequence length (dim1) remained at 12


## 11. Summary <a id='summary'></a>

### Key Takeaways

1. **Purpose**: `SampleMetadata` captures and tracks metadata about tensors in complex data structures, enabling model optimization and dynamic batching.

2. **Creation**: Use `SampleMetadata.from_inputs(inputs, strict=bool)` with inputs keyed by forward parameter name.

3. **Strict Mode**: Controls whether only tensors (`strict=False`) or all data types (`strict=True`) are tracked.

4. **Nested Structures**: Automatically handles tuples, lists, dicts, and dataclasses using Locators.

5. **Parameter Paths**: Locators identify tensor leaves by their forward parameter paths.

6. **InfoLevel**: Three display modes (SHORT, MEDIUM, FULL) provide different levels of detail.

7. **Dynamic Shape Learning**: `update_shapes_seen()` learns about batch axes and dynamic dimensions by observing multiple samples.

8. **Batch Manipulation**: `make_batch()` can scale tensors to any batch size based on learned batch axis multipliers.

9. **TensorSpec**: Stores shape, dtype, and batch information; its paired Locator provides tensor identity.

### Use in AI-Tune Pipeline

`SampleMetadata` is a fundamental building block in the ai-tune library, used by:

- **RecordingModule**: Captures input/output metadata during profiling
- **Backend Adapters**: Use metadata to configure optimized execution (TensorRT, TorchScript, etc.)
- **Graph Compilation**: Enables creation of optimized graphs for dynamic shapes
- **Memory Planning**: Helps allocate appropriate buffers based on observed ranges

### Source Code

For more details, see:
- `aitune/torch/module/sample_metadata.py`
- `aitune/torch/module/tensor_spec.py`
- `aitune/torch/module/locator.py`
